# ORCA PFZ ML - Notebook 01: Data Download & Verification

**Purpose**: Verify access to real satellite datasets and download for ML training.

**Data Sources**:
1. NOAA OISST v2.1 - Sea Surface Temperature (no auth required)
2. NASA MODIS-Aqua L3 - Chlorophyll-a (free Earthdata login required)

**IMPORTANT**: This notebook downloads REAL satellite data from official sources.
Do NOT replace with synthetic/fabricated data.

## Step 0: Install Dependencies

In [ ]:
!pip install xarray netCDF4 matplotlib cartopy numpy pandas tqdm

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import urllib.request
import os
from datetime import datetime, timedelta
from pathlib import Path
from tqdm import tqdm

print(f"xarray version: {xr.__version__}")
print(f"numpy version: {np.__version__}")

## Step 1: Verify NOAA OISST Access (Single File Test)

Source: https://www.ncei.noaa.gov/data/sea-surface-temperature-optimum-interpolation/v2.1/access/avhrr/

- Resolution: 0.25 deg (~28km), daily
- Format: NetCDF
- No authentication required

In [ ]:
# Download a single NOAA OISST file to verify accessibility
SST_BASE_URL = "https://www.ncei.noaa.gov/data/sea-surface-temperature-optimum-interpolation/v2.1/access/avhrr"

test_date = "20200101"
test_month = "202001"
test_filename = f"oisst-avhrr-v02r01.{test_date}.nc"
test_url = f"{SST_BASE_URL}/{test_month}/{test_filename}"

print(f"Attempting download from:\n{test_url}")
print("\nThis requires NO authentication...")

output_path = f"/content/{test_filename}"

try:
    urllib.request.urlretrieve(test_url, output_path)
    file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"\nSUCCESS: Downloaded {test_filename}")
    print(f"File size: {file_size_mb:.2f} MB")
except Exception as e:
    print(f"\nFAILED: {e}")
    print("\nTry alternative: OPeNDAP access or different NCEI endpoint")

In [ ]:
# Open and inspect the downloaded SST file
ds = xr.open_dataset(output_path)
print("=" * 60)
print("NOAA OISST v2.1 - Dataset Structure")
print("=" * 60)
print(ds)
print("\n" + "=" * 60)
print("Variables:")
print("=" * 60)
for var in ds.data_vars:
    print(f"  {var}: {ds[var].dims} | {ds[var].dtype} | {ds[var].attrs.get('long_name', '')}")

In [ ]:
# Inspect SST variable details
print("SST Variable Attributes:")
for key, val in ds['sst'].attrs.items():
    print(f"  {key}: {val}")

print(f"\nCoordinate Ranges:")
print(f"  Latitude:  {float(ds.lat.min()):.2f} to {float(ds.lat.max()):.2f}")
print(f"  Longitude: {float(ds.lon.min()):.2f} to {float(ds.lon.max()):.2f}")
print(f"  Time:      {ds.time.values}")

In [ ]:
# Subset to Indian Ocean region
# Indian coastal waters: Lat 7N-23N, Lon 66E-95E
indian_ocean = ds.sel(lat=slice(7, 23), lon=slice(66, 95))

print("Indian Ocean Subset:")
print(f"  Shape: {indian_ocean['sst'].shape}")
print(f"  Grid cells: {indian_ocean['sst'].shape[-2]} lat x {indian_ocean['sst'].shape[-1]} lon")

sst_data = indian_ocean['sst'].isel(time=0, zlev=0)
valid_data = sst_data.values[~np.isnan(sst_data.values)]

print(f"\nSST Statistics (Indian Ocean):")
print(f"  Min: {np.nanmin(valid_data):.2f} deg C")
print(f"  Max: {np.nanmax(valid_data):.2f} deg C")
print(f"  Mean: {np.nanmean(valid_data):.2f} deg C")
print(f"  Std: {np.nanstd(valid_data):.2f} deg C")
print(f"  Valid pixels: {len(valid_data)}")
print(f"  NaN pixels (land): {np.isnan(sst_data.values).sum()}")

In [ ]:
# Visualize SST for Indian Ocean
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

sst_plot = indian_ocean['sst'].isel(time=0, zlev=0)
im = sst_plot.plot(ax=ax, cmap='RdYlBu_r', vmin=20, vmax=32,
                   cbar_kwargs={'label': 'SST (deg C)'})

ax.set_title('NOAA OISST v2.1 - Indian Ocean\n2020-01-01', fontsize=14)
ax.set_xlabel('Longitude (E)')
ax.set_ylabel('Latitude (N)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/sst_indian_ocean_verification.png', dpi=150)
plt.show()

print("\nIf you can see the Indian Ocean SST map above, data access is VERIFIED.")

## Step 2: Verify NASA MODIS Chlorophyll Access

Source: https://oceandata.sci.gsfc.nasa.gov/

**Requires**: Free NASA Earthdata account
- Register at: https://urs.earthdata.nasa.gov/
- After registration, you need to authorize the OB.DAAC application

If you haven't registered yet, do that FIRST, then come back to this cell.

In [ ]:
# NASA Earthdata credentials (replace with your own after registering)
# DO NOT commit real credentials to git

EARTHDATA_USERNAME = "YOUR_USERNAME_HERE"  # Replace after registering
EARTHDATA_PASSWORD = "YOUR_PASSWORD_HERE"  # Replace after registering

if EARTHDATA_USERNAME == "YOUR_USERNAME_HERE":
    print("WARNING: You need to register at https://urs.earthdata.nasa.gov/")
    print("Then replace the username and password above.")
    print("\nRegistration is FREE and takes ~2 minutes.")
    print("\nAfter registering:")
    print("1. Go to https://urs.earthdata.nasa.gov/")
    print("2. Log in")
    print("3. Go to Applications -> Authorized Apps")
    print("4. Search for and authorize 'OB.DAAC'")
else:
    print(f"Earthdata credentials configured for user: {EARTHDATA_USERNAME}")

In [ ]:
# Set up authenticated download for NASA MODIS data
# Using .netrc file approach (standard NASA Earthdata method)

import os

if EARTHDATA_USERNAME != "YOUR_USERNAME_HERE":
    netrc_path = os.path.expanduser("~/.netrc")
    with open(netrc_path, 'w') as f:
        f.write(f"machine urs.earthdata.nasa.gov login {EARTHDATA_USERNAME} password {EARTHDATA_PASSWORD}\n")
    os.chmod(netrc_path, 0o600)
    print("Created ~/.netrc for NASA Earthdata authentication")
    
    # Test download of one MODIS chlorophyll file
    # 8-day composite, 4km resolution, January 2020
    modis_url = "https://oceandata.sci.gsfc.nasa.gov/ob/getfile/AQUA_MODIS.20200101_20200108.L3m.8D.CHL.chlor_a.4km.nc"
    
    print(f"\nAttempting MODIS download: {modis_url}")
    
    import subprocess
    result = subprocess.run(
        ['wget', '--quiet', '--no-check-certificate', '-nc', 
         '--auth-no-challenge=on', modis_url, '-O', '/content/test_chlorophyll.nc'],
        capture_output=True, text=True
    )
    
    if os.path.exists('/content/test_chlorophyll.nc') and os.path.getsize('/content/test_chlorophyll.nc') > 1000:
        print(f"SUCCESS: Downloaded MODIS chlorophyll file")
        print(f"Size: {os.path.getsize('/content/test_chlorophyll.nc') / (1024*1024):.2f} MB")
    else:
        print(f"Download may have failed. Check credentials and OB.DAAC authorization.")
        print(f"wget stderr: {result.stderr}")
else:
    print("Skipping MODIS download - register for NASA Earthdata first.")
    print("URL: https://urs.earthdata.nasa.gov/")

In [ ]:
# If MODIS download succeeded, inspect the chlorophyll data
chl_path = '/content/test_chlorophyll.nc'

if os.path.exists(chl_path) and os.path.getsize(chl_path) > 1000:
    ds_chl = xr.open_dataset(chl_path)
    print("=" * 60)
    print("NASA MODIS-Aqua - Chlorophyll-a Dataset Structure")
    print("=" * 60)
    print(ds_chl)
    
    # Subset to Indian Ocean
    indian_chl = ds_chl.sel(lat=slice(23, 7), lon=slice(66, 95))  # Note: lat may be descending
    
    print(f"\nIndian Ocean Chlorophyll-a Stats:")
    chl_vals = indian_chl['chlor_a'].values.flatten()
    chl_valid = chl_vals[~np.isnan(chl_vals)]
    print(f"  Valid pixels: {len(chl_valid)}")
    print(f"  Min: {np.min(chl_valid):.4f} mg/m3")
    print(f"  Max: {np.max(chl_valid):.4f} mg/m3")
    print(f"  Mean: {np.mean(chl_valid):.4f} mg/m3")
    print(f"  Median: {np.median(chl_valid):.4f} mg/m3")
else:
    print("MODIS file not available. Register at https://urs.earthdata.nasa.gov/ first.")
    print("\nAlternative: Try OPeNDAP or ERDDAP access for chlorophyll data.")

## Step 3: Batch Download NOAA OISST (After Verification)

Only run this after Step 1 verification succeeds.

Downloads daily SST for 2020-2024 to Google Drive.

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Create storage directories
SST_DIR = '/content/drive/MyDrive/orca_ml/raw/sst'
CHL_DIR = '/content/drive/MyDrive/orca_ml/raw/chlorophyll'
os.makedirs(SST_DIR, exist_ok=True)
os.makedirs(CHL_DIR, exist_ok=True)

print(f"SST storage: {SST_DIR}")
print(f"Chlorophyll storage: {CHL_DIR}")

In [ ]:
# Batch download NOAA OISST for training period
# Training: 2020-01-01 to 2024-12-31

from datetime import date

start_date = date(2020, 1, 1)
end_date = date(2024, 12, 31)

# Generate all dates
current = start_date
dates_to_download = []
while current <= end_date:
    dates_to_download.append(current)
    current += timedelta(days=1)

print(f"Total files to download: {len(dates_to_download)}")
print(f"Estimated size: {len(dates_to_download) * 1.7:.0f} MB ({len(dates_to_download) * 1.7 / 1024:.1f} GB)")
print(f"\nDate range: {start_date} to {end_date}")
print(f"\nNote: This will take significant time. Consider downloading in chunks.")

In [ ]:
# Download SST files (with progress bar and error handling)
# WARNING: This downloads ~3GB. Run in chunks if needed.

failed_downloads = []
successful_downloads = 0
skipped = 0

# Download first month as test (uncomment the full loop for all data)
# To download all: change end_subset to end_date
end_subset = date(2020, 1, 31)  # Start with just January 2020
subset_dates = [d for d in dates_to_download if d <= end_subset]

print(f"Downloading {len(subset_dates)} files (subset for testing)...")
print(f"Change end_subset to download full dataset.\n")

for d in tqdm(subset_dates, desc="Downloading SST"):
    filename = f"oisst-avhrr-v02r01.{d.strftime('%Y%m%d')}.nc"
    filepath = os.path.join(SST_DIR, filename)
    
    # Skip if already downloaded
    if os.path.exists(filepath) and os.path.getsize(filepath) > 100000:
        skipped += 1
        continue
    
    url = f"{SST_BASE_URL}/{d.strftime('%Y%m')}/{filename}"
    
    try:
        urllib.request.urlretrieve(url, filepath)
        successful_downloads += 1
    except Exception as e:
        failed_downloads.append((d, str(e)))

print(f"\nDownload Summary:")
print(f"  Successful: {successful_downloads}")
print(f"  Skipped (already exists): {skipped}")
print(f"  Failed: {len(failed_downloads)}")

if failed_downloads:
    print(f"\nFailed dates:")
    for d, err in failed_downloads[:10]:
        print(f"  {d}: {err}")

## Step 4: Data Quality Verification

After downloading, verify data integrity.

In [ ]:
# Verify downloaded SST files
sst_files = sorted([f for f in os.listdir(SST_DIR) if f.endswith('.nc')])

print(f"Total SST files on Drive: {len(sst_files)}")

if sst_files:
    print(f"First file: {sst_files[0]}")
    print(f"Last file: {sst_files[-1]}")
    
    # Verify a random file can be opened
    import random
    sample_file = os.path.join(SST_DIR, random.choice(sst_files))
    ds_check = xr.open_dataset(sample_file)
    
    indian_check = ds_check.sel(lat=slice(7, 23), lon=slice(66, 95))
    sst_vals = indian_check['sst'].values.flatten()
    valid = sst_vals[~np.isnan(sst_vals)]
    
    print(f"\nSample file verification ({os.path.basename(sample_file)}):")
    print(f"  Indian Ocean valid pixels: {len(valid)}")
    print(f"  SST range: {valid.min():.2f} to {valid.max():.2f} deg C")
    
    # Check for physically impossible values
    impossible = (valid < -2) | (valid > 40)
    print(f"  Impossible values (< -2C or > 40C): {impossible.sum()}")
    
    ds_check.close()

## Step 5: Report — Data Discovery Status

Run this cell to generate a summary of what data is available.

In [ ]:
print("=" * 70)
print("ORCA PFZ ML — DATA DISCOVERY REPORT")
print("=" * 70)

print("\n1. NOAA OISST v2.1 (SST)")
print("-" * 40)
if os.path.exists(output_path):
    print("   Status: ACCESSIBLE (verified)")
    print(f"   Test file: {test_filename}")
    print(f"   Resolution: 0.25 deg daily")
    print(f"   Auth: None required")
    print(f"   Indian Ocean grid: ~64 lat x ~116 lon cells")
else:
    print("   Status: NOT VERIFIED")

print("\n2. NASA MODIS Chlorophyll-a")
print("-" * 40)
if os.path.exists(chl_path) and os.path.getsize(chl_path) > 1000:
    print("   Status: ACCESSIBLE (verified)")
    print("   Resolution: 4km 8-day composite")
    print("   Auth: NASA Earthdata (configured)")
else:
    print("   Status: PENDING (requires NASA Earthdata registration)")
    print("   Action: Register at https://urs.earthdata.nasa.gov/")

print("\n3. INCOIS Direct Data")
print("-" * 40)
print("   Status: NOT DIRECTLY ACCESSIBLE")
print("   Issue: SSL cert errors, 404 on data portal URLs")
print("   Alternative: MOSDAC registration (mosdac.gov.in)")
print("   Action: Attempt manual registration on MOSDAC")

print("\n4. PFZ Advisory Archives")
print("-" * 40)
print("   Status: NOT PROGRAMMATICALLY ACCESSIBLE")
print("   Plan: Use pseudo-labels from SST gradient + chlorophyll thresholds")
print("   Validation: Compare against published INCOIS PFZ maps if obtainable")

print("\n" + "=" * 70)
print("NEXT STEPS:")
print("=" * 70)
print("1. If OISST verified: Proceed to batch download (Step 3)")
print("2. Register NASA Earthdata and download chlorophyll")
print("3. Attempt MOSDAC registration for Indian satellite products")
print("4. Move to Notebook 02 for preprocessing once data is acquired")
print("\nDO NOT proceed to model training until both SST and Chlorophyll")
print("datasets are downloaded and verified.")